# 02 - Training Experiments

Notebook principal para experimentar entrenamientos y registrar en W&B.

Tip:
- Raul: usa `configs/experiments/gtx1650.yaml`
- Natalia: usa `configs/experiments/rtx3070.yaml`

In [ ]:
from pathlib import Path
import sys

import torch
import wandb

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from real_estate_ml.config import load_config
from real_estate_ml.constants import CLASSES
from real_estate_ml.data.dataset import get_dataloaders
from real_estate_ml.models.classifier import build_model
from real_estate_ml.training.engine import run_epoch

In [ ]:
CONFIG_PATH = ROOT / "configs" / "experiments" / "gtx1650.yaml"
# CONFIG_PATH = ROOT / "configs" / "experiments" / "rtx3070.yaml"

cfg = load_config(CONFIG_PATH)
cfg

In [ ]:
device = torch.device("cuda" if cfg["hardware"]["device"] == "cuda" and torch.cuda.is_available() else "cpu")
print("Using device:", device)

dataloaders = get_dataloaders(
    data_dir=str(ROOT / cfg["data"]["data_dir"]),
    batch_size=cfg["data"]["batch_size"],
    image_size=cfg["data"]["image_size"],
    num_workers=cfg["data"]["num_workers"],
)

model = build_model(
    backbone=cfg["model"]["backbone"],
    num_classes=cfg["data"]["num_classes"],
    pretrained=cfg["model"]["pretrained"],
    dropout=cfg["model"]["dropout"],
    freeze_backbone=cfg["model"]["freeze_backbone"],
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg["training"]["learning_rate"]),
    weight_decay=float(cfg["training"]["weight_decay"]),
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["training"]["epochs"])

run = wandb.init(project=cfg["project_name"], entity=cfg["entity"], config=cfg, job_type="train-notebook")

In [ ]:
best_macro_f1 = -1.0
patience = 0
best_model_path = ROOT / cfg["training"]["save_dir"] / "best_model.pth"
best_model_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(cfg["training"]["epochs"]):
    train_metrics = run_epoch(model, dataloaders["train"], criterion, optimizer, device, train=True)
    val_metrics = run_epoch(model, dataloaders["val"], criterion, optimizer, device, train=False)
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_metrics.loss,
        "train/macro_f1": train_metrics.macro_f1,
        "val/loss": val_metrics.loss,
        "val/macro_f1": val_metrics.macro_f1,
    })

    print(f"Epoch {epoch + 1}: train_f1={train_metrics.macro_f1:.4f}, val_f1={val_metrics.macro_f1:.4f}")

    if val_metrics.macro_f1 > best_macro_f1:
        best_macro_f1 = val_metrics.macro_f1
        patience = 0
        torch.save({"model_state_dict": model.state_dict()}, best_model_path)
    else:
        patience += 1

    if patience >= cfg["training"]["early_stopping_patience"]:
        print("Early stopping")
        break

print("Best val macro_f1:", round(best_macro_f1, 4))

In [ ]:
artifact = wandb.Artifact("best-model", type="model")
artifact.add_file(str(best_model_path))
run.log_artifact(artifact)
run.finish()

best_model_path